# Evaluación reproducible y equivalencia de inferencia

Datos AI4I 2020 sintéticos (UCI, DOI 10.24432/C5HS5C, CC BY 4.0). Clasificación de condición actual, sin horizonte futuro. Solo se usan las seis características publicadas. El código de este notebook invoca el mismo entrenamiento que la CLI; la API se comprueba en proceso con TestClient.

In [ ]:
from pathlib import Path
import pandas as pd
from fastapi.testclient import TestClient
from predictive_monitor.training import train, read_dataset
from predictive_monitor.inference import Predictor
from predictive_monitor.schema import Observation
from predictive_monitor.api import create_app
root = Path.cwd()
if not (root / "data/raw/ai4i2020.csv").exists():
    root = root.parent
artifact = root / "var/notebook-model"
report = train(root / "data/raw/ai4i2020.csv", artifact)
report["partitions"]

Se conserva el orden UDI en bloques 60/20/20. No hay fechas ni grupos de activos acreditados. El umbral se elige por F2 únicamente en validación. Las métricas de prueba no ajustan el modelo.

In [ ]:
pd.DataFrame(report["evaluation"]["test"]).T

In [ ]:
x, y, ids = read_dataset(root / "data/raw/ai4i2020.csv")
sample = Observation(**x.iloc[8000].to_dict())
direct = Predictor(artifact).predict(sample)
with TestClient(create_app(artifact, root / "var/notebook.db")) as client:
    response = client.post("/predict", json=sample.model_dump())
    assert response.status_code == 200, response.text
    assert response.json() == direct
direct

La igualdad anterior valida el recorrido dentro del proceso. No es una medida de latencia HTTP ni una prueba de integración con equipos físicos. Consultar el informe de modelo para falsos negativos, falsas alarmas y limitaciones del dataset.